# Notebook 14 — Image Foundation Models and Embeddings
Goal: Use a pretrained vision model to convert images into embeddings and analyze similarity.

This notebook introduces the pattern:

`image → pretrained vision model → embedding → similarity / clustering`


## Section 1 — Install and Import Libraries

In [ ]:
# If needed:
# pip install torch torchvision transformers scikit-learn matplotlib pillow

import torch
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from transformers import AutoImageProcessor, AutoModel
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

## Section 2 — Load a Pretrained Vision Model
We will use a small Vision Transformer (ViT) model.

In [ ]:
model_name = "google/vit-base-patch16-224"

processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model_name

## Section 3 — Create Example Images
To keep the notebook self-contained we will generate simple synthetic images.

In [ ]:
images = []

for i in range(6):
    img = np.zeros((224,224,3))
    img[:, :, :] = np.random.rand(1) * 0.5
    
    if i % 2 == 0:
        img[80:140, 80:140] = [1,0,0]
    else:
        img[60:160, 60:160] = [0,0,1]
        
    images.append((img * 255).astype(np.uint8))

len(images)

In [ ]:
plt.figure(figsize=(8,4))

for i,img in enumerate(images):
    plt.subplot(2,3,i+1)
    plt.imshow(img)
    plt.axis("off")

plt.suptitle("Example Images")
plt.show()

## Section 4 — Convert Images to Embeddings

In [ ]:
def get_image_embedding(img):
    inputs = processor(images=img, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).squeeze().numpy()

embeddings = [get_image_embedding(img) for img in images]

len(embeddings), embeddings[0].shape

## Section 5 — Compute Similarity Between Images

In [ ]:
sim = cosine_similarity(embeddings)

sim.round(3)

## Section 6 — Visualize Embeddings with PCA

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(embeddings)

plt.figure(figsize=(6,5))
plt.scatter(coords[:,0], coords[:,1])

for i,(x,y) in enumerate(coords):
    plt.text(x,y,str(i))

plt.title("PCA of Image Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

## Section 7 — Interpretation
Images with similar structure should have embeddings closer together.
This is the same idea used in many microscopy foundation-model workflows.

## Section 8 — Exercises

1. Modify the synthetic images (different shapes or colors).
2. Regenerate embeddings and check whether similarity changes.
3. Add more images and repeat the PCA visualization.
4. Identify the two most similar images.
5. Compute cosine similarity between two specific images.

## Skills Gained
- using a pretrained vision transformer
- converting images into embeddings
- comparing image embeddings
- visualizing embedding space
- understanding how vision foundation models represent images